# సవాలు: డేటా సైన్స్ గురించి పాఠ్యం విశ్లేషణ

ఈ ఉదాహరణలో, సంప్రదాయ డేటా సైన్స్ ప్రక్రియ అన్ని దశలను కవర్ చేసే ఒక సులభమైన వ్యాయామం చేద్దాం. మీరు ఎలాంటి కోడ్ రాయాల్సిన అవసరం లేదు, కింది సెల్‌లపై క్లిక్ చేసి వాటిని అమలు చేయవచ్చు మరియు ఫలితాన్ని గమనించవచ్చు. ఒక సవాలుగా, మీరు ఈ కోడ్‌ను వేరే డేటాతో ప్రయత్నించాలని ప్రోత్సహించబడతారు.

## లక్ష్యం

ఈ పాఠంలో, మేము డేటా సైన్స్‌కు సంబంధించిన విభిన్న భావనలు చర్చిస్తున్నాము. కొంత **పాఠ్య తవ్వకం** చేయడం ద్వారా మరిన్ని సంబంధిత భావనలు కనుగొనడానికి ప్రయత్నిద్దాం. మేము డేటా సైన్స్ గురించి ఒక పాఠ్యంతో మొదలుపెట్టి, దాని నుండి కీలక పదాల‌ను తీసుకుని, తర్వాత ఫలితాన్ని దృశ్యమానంగా చూపించడానికి ప్రయత్నిస్తాం.

పాఠ్యంగా నేను వికీపీడియా నుండి డేటా సైన్స్ పేజీని ఉపయోగిస్తాను:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## దశ 1: డేటాను పొందడం

ప్రతి డేటా సైన్స్ ప్రక్రియలో మొదటి దశ డేటాను పొందడం. దీని కోసం మేము `requests` లైబ్రరీని ఉపయోగించబోతున్నాం:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## దశ 2: డేటాను మార్చడం

తర్వాతి దశగా డేటాను ప్రాసెస్‍చేయడానికి అనుకూలమైన రూపంగా మార్చడం. మనం పేజీ నుండి HTML సోర్స్ కోడ్‌ను డౌన్‌లోడ్ చేసుకున్నాము, దీన్ని ప్లెయిన్ టెక్స్ట్‌గా మార్చాలి.

ఇది చేయడానికి అనేక మార్గాలు ఉన్నాయి. మనం [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/) ఉపయోగిస్తాము, ఇది HTML పార్సింగ్ కోసం ప్రాచుర్యం పొందిన Python లైబ్రరీ. BeautifulSoup మాకు నిర్దిష్ట HTML అంశాలను లక్ష్యం చేయడానికీ అవకాశం ఇస్తుంది, అందుచే మనం Wikipedia యొక్క ప్రధాన వ్యాసం కంటెంట్‌పై దృష్టి పెట్టవచ్చు, మరియు కొన్ని నావిగేషన్ మెనూలు, సైడ్ బార్‌లు, ఫూటర్లు మరియు ఇతర ప్రాసంగికం కాని భాగాలను తగ్గించవచ్చు (యాదృచ్ఛికంగా కొన్ని బోయిలర్‌ప్లేట్ టెక్స్ట్ ఇంకా ఉండొచ్చు).  


మొదట, మనం HTML పార్సింగ్ కోసం BeautifulSoup గ్రంథాలయాన్ని ఇన్‌స్టాల్ చేయవలసింది:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## దశ 3: అవగాహన పొందడం

అత్యంత ముఖ్యమైన దశ ఏమిటంటే మన డేటాను అలాంటి రూపంలోకి మార్చడం, దాని నుండి మనం అవగాహన పొందగలుగుతాము. మన సందర్భంలో, మనం టెక్స్ట్ నుండి కీలకపదాలను తీసుకోవాలనుకుంటున్నాం, మరియు ఏ కీలకపదాలు మరింత అర్ధం గలవో చూడాలి.

మనం పాథాన్ లైబ్రరీ [RAKE](https://github.com/aneesha/RAKE)ను కీలకపదాల తీసుకోవడానికి ఉపయోగించబడతాం. మొదట, ఇది తగ్గి ఉంటే ఈ లైబ్రరీని ఇన్‌స్టాల్ చేసుకోదాం: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

ప్రధాన ఫంక్షనాలిటీ `Rake` ఆబ్జెక్ట్ నుండి అందుబాటులో ఉంటుంది, దీన్ని మేము కొన్ని పారామితులు ఉపయోగించి అనుకూలీకరించుకోవచ్చు. మా సందర్భంలో, ఒక కీవర్డ్ కనిష్ఠదైర్ఘ్యం 5 అక్షరాలు గా, డాక్యుమెంట్ లో కీవర్డ్ కనిష్ఠ సాంద్రత 3 గా, మరియు కీవర్డ్ లో గరిష్ఠ పదాల సంఖ్య - 2 గా సెట్ చేస్తాము. ఇతర విలువలతో ప్రయోగించి ఫలితాన్ని పరిశీలించండి.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


మేము సంబంధించిన ప్రాధాన్యత డిగ్రీతో పాటు పదాల జాబితాను పొందాము. మీరు చూడగలిగినట్లుగా, యంత్రమైన అభ్యాసం మరియు పెద్ద డేటా వంటి అత్యంత సంబంధిత శాస్త్రాలు జాబితాలో పై స్థానాల్లో ఉన్నాయి.

## దశ 4: ఫలితాన్ని దృశ్యరూపంలో చూపించడం

డేటాను అందరికీ బాగా అర్థం కావడానికి దృశ్యరూపం ఉత్తమమైంది. అందుకే కొన్ని వివరాలు పొందటానికి డేటాను దృశ్యరూపంలో చూపించడం సాధారణం. పదసమూహాల సంబంధాన్ని సులువుగా చూపించేందుకు Python లో `matplotlib` లైబ్రరీని ఉపయోగించవచ్చు:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

అయితే, పదపు ఆధారాల సరళిని చూడడానికి మరొక మంచి విధానం ఉంది - **Word Cloud** ఉపయోగించడం. మా కీవర్డ్ జాబితా నుండి పదపు మేఘాన్ని చిత్రీకరించడానికి మేము మరొక లైబ్రరీని ఇన్స్టాల్ చేయాలి.


In [ ]:
!{sys.executable} -m pip install wordcloud

`WordCloud` ఆబ్జెక్ట్ అసలు పాఠ్యం లేదా పదాలతో కూడిన వారి ఫ్రీక్వెన్సీలతో కూడిన ముందుగా లెక్కించబడిన జాబితాను తీసుకొని, ఒక చిత్రాన్ని తిరిగి ఇస్తుంది, దీనిని తరువాత `matplotlib` ఉపయోగించి ప్రదర్శించవచ్చు:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

మేము అసలు వచనాన్ని కూడా `WordCloud` కి పాస్ చేయవచ్చు - మేము సమాన ఫలితాన్ని పొందగలమో చూడండి:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

మీరు ఇప్పుడు ఆ పద మేఘం మరింత ఆకర్షణీయంగా కనిపిస్తోంది అయితే, ఇది చాలా శబ్దం కూడా కలిగి ఉంది (ఉదా: `Retrieved on` వంటి సంబంధం లేని పదాలు). ఇంకా, మనం రెండు పదాల కలయిక నుండి ఉన్న తక్కువ కీవర్డ్స్ మాత్రమే పొందుతున్నాము, ఉదాహరణకు *డేటా సైంటిస్ట్*, లేదా *కంప్యూటర్ సైన్స్*. ఇది ఎందుకంటే RAKE ఆల్గోరిథం టెక్ట్స్ నుండి మంచి కీవర్డ్స్ ఎంచుకోవడంలో బాగా పనిచేస్తుంది. ఈ ఉదాహరణ డేటా ప్రీ-ప్రాసెసింగ్ మరియు శుభ్రపరిచే ప్రక్రియ యొక్క ప్రాముఖ్యతను స్పష్టం చేస్తుంది, ఎందుకంటే చివరికి స్పష్టమైన చిత్రం మనకు మెరుగైన నిర్ణయాలు తీసుకోవడానికి అనుమతిస్తుంది.

ఈ వ్యాయామంలో మనం వికీపీడియా టెక్ట్స్ నుండి కొంత అర్థం çıkar గడానికి ఒక సులభమైన ప్రక్రియను పూర్తిచేశాము, కీవర్డ్స్ మరియు పద మేఘం రూపంలో. ఈ ఉదాహరణ సులభమైనది, కానీ ఇది డేటా సైంటిస్ట్ డేటాతో పని చేసేటప్పుడు తీసుకునే అన్ని సాధారణ దశలను బాగా చూపిస్తోంది, డేటా సేకరణ నుండి ప్రారంభమై విజువలైజేషన్ వరకు.

మన కోర్సులో ఆ దశలన్నింటినీ వివరంగా చర్చిస్తాము.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
